# Aset Solana

In [ ]:
!pip install blitz-bayesian-pytorch

## One-Time Code

In [ ]:
# Import Library

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from datetime import datetime, timedelta
import pickle
import os
import sys
from tqdm.auto import tqdm 
from scipy import stats

# ==============================================================================
# 0. IMPORT & LIBRARY CHECK
# ==============================================================================
try:
    import yfinance as yf
    from blitz.modules import BayesianLSTM, BayesianLinear
    from blitz.utils import variational_estimator
except ImportError:
    print("❌ ERROR: Library belum lengkap.")
    print("👉 Jalankan: pip install blitz-bayesian-pytorch yfinance pandas numpy matplotlib scikit-learn torch scipy tqdm")
    sys.exit(1)

In [ ]:
# ==============================================================================
# 1. KONFIGURASI & HIPERPARAMETER
# ==============================================================================
CONFIG = {
    'ticker': 'SOL-USD',    
    'seed': 42,
    'seq_length': 168,      # Input Horizon (1 Minggu)
    'batch_size': 64,
    'epochs': 100,           # Jumlah Epoch
    'lr': 0.001,            # Learning Rate
    'hidden_size_1': 32,
    'hidden_size_2': 16,
    'n_samples': 50,        # Jumlah Monte Carlo Sampling
    'risk_free_rate': 0.0372, # Risk Free Rate (3,72% US Monthly Bill)
    'start_date': '2021-01-01',
    'end_date': '2025-11-30',
    'test_start_date': '2025-01-01'
}

MODEL_PATH = "bayesian_lstm_model.pth"
ARTIFACTS_PATH = "training_artifacts.pkl"

# Set Seed untuk Reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# HELPER FUNCTION

def download_and_preprocess_data(ticker, start_date, end_date):
    """
    Mengambil data dari input Kaggle.
    """
    print(f"🔄 Mengunduh {ticker} ({start_date} s/d {end_date})...")
    path = "/kaggle/input/dataset-tesis/SOL.csv"
    df = pd.read_csv(path)
    # Pastikan kolom 'Date' ada dan set sebagai index
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)

    required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    if not all(col in df.columns for col in required_cols):
             raise ValueError(f"CSV tidak lengkap. Kolom wajib: {required_cols}. Kolom tersedia: {df.columns}")
        
    df = df[required_cols].copy()

    # Potong ekor data yang mungkin kelebihan (misal masuk Des 2025)
    df = df[df.index <= end_date]

    # RESAMPLING & FILLING 
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None) # Hapus timezone
        
    df = df[~df.index.duplicated(keep='first')] # Hapus duplikat
    df = df.resample('1h').asfreq() # Paksa index jadi hourly sequence
    df = df.ffill().bfill() # Tambal data kosong

    # Print informasi dataset
    print(f"✅ Data Siap: {len(df)} baris (Full Hour Sequence).")
    print(f"   Start: {df.index[0]}")
    print(f"   End  : {df.index[-1]}")
    
    return df

def feature_engineering(df):
    """Complex Feature Engineering for Stationary Time Series"""
    df_feat = df.copy()
    # Log Returns & Volatility Metrics (Ditambah epsilon kecil 1e-9 agar tidak log(0))
    df_feat['log_return'] = np.log(df_feat['Close'] / df_feat['Close'].shift(1) + 1e-9)
    df_feat['log_range'] = np.log(df_feat['High'] / df_feat['Low'] + 1e-9)
    df_feat['log_body'] = np.log(df_feat['Close'] / df_feat['Open'] + 1e-9)
    df_feat['log_vol_change'] = np.log((df_feat['Volume'] + 1.0) / (df_feat['Volume'].shift(1) + 1.0))
    
    # Cyclical Time Encoding
    hours = df_feat.index.hour
    days = df_feat.index.dayofweek
    df_feat['hour_sin'] = np.sin(2 * np.pi * hours / 24)
    df_feat['hour_cos'] = np.cos(2 * np.pi * hours / 24)
    df_feat['day_sin'] = np.sin(2 * np.pi * days / 7)
    df_feat['day_cos'] = np.cos(2 * np.pi * days / 7)
    
    df_feat.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_feat.dropna(inplace=True)
    return df_feat

class CryptoDataset(Dataset):
    def __init__(self, X, y, seq_length=168):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.seq_length = seq_length
    def __len__(self): return len(self.X) - self.seq_length
    def __getitem__(self, i): return self.X[i:i+self.seq_length], self.y[i+self.seq_length]

# ==============================================================================
# 3. MODEL ARSITEKTUR (Bayesian LSTM)
# ==============================================================================
@variational_estimator
class BayesianLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_1, hidden_2, output_dim=2):
        super().__init__()
        self.lstm1 = BayesianLSTM(input_dim, hidden_1, prior_sigma_1=1, prior_pi=1, posterior_rho_init=-3.0)
        self.lstm2 = BayesianLSTM(hidden_1, hidden_2, prior_sigma_1=1, prior_pi=1, posterior_rho_init=-3.0)
        self.linear = BayesianLinear(hidden_2, output_dim, prior_sigma_1=1, prior_pi=1, posterior_rho_init=-3.0)
        self.act = nn.Tanh()
        
    def forward(self, x):
        out1, _ = self.lstm1(x); out1 = self.act(out1)
        out2, _ = self.lstm2(out1); last_hidden = out2[:, -1, :]
        params = self.linear(last_hidden)
        # Output: [Mean, Raw_Sigma] 
        mu, sigma = params[:, 0], nn.functional.softplus(params[:, 1]) + 1e-6
        return mu, sigma

# ==============================================================================
# 4. TRAINING ENGINE
# ==============================================================================
def train_engine(model, train_loader, val_loader, epochs, device, lr):
    # 1. SETUP OPTIMIZER & LOSS
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.GaussianNLLLoss()
    history = {'train_loss': [], 'val_loss': []}

    # --- TAMBAHAN SCHEDULER ---
    # Kalau val_loss macet 5 epoch, kurangi LR jadi setengahnya
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
    )

    # 2. KONFIGURASI COLD POSTERIOR
    kl_weight_start = 0.0
    kl_weight_end = 0.01   
    burn_in_epochs = 10    
    anneal_epochs = 40     

    # 3. SETUP EARLY STOPPING 
    best_val_loss = float('inf') 
    patience = 15                
    counter = 0                  

    print(f"🚀 Mulai Training ({epochs} Epochs) dengan Soft Annealing...")

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        # --- LOGIKA KL WEIGHT ---
        if epoch < burn_in_epochs:
            kl_weight = 0.0
        elif epoch < (burn_in_epochs + anneal_epochs):
            progress = (epoch - burn_in_epochs) / anneal_epochs
            kl_weight = kl_weight_start + progress * (kl_weight_end - kl_weight_start)
        else:
            kl_weight = kl_weight_end 

        # --- TRAINING LOOP ---
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            mu, sigma = model(X_batch)
            
            nll_loss = criterion(mu, y_batch, sigma.pow(2)) 
            kl_loss = model.nn_kl_divergence()
            
            scaling_factor = len(train_loader.dataset)
            loss = nll_loss + (kl_weight * (kl_loss / scaling_factor))
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- VALIDATION LOOP ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                mu_val, sigma_val = model(X_val)
                val_loss += criterion(mu_val, y_val, sigma_val.pow(2)).item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)

        # --- UPDATE SCHEDULER ---
        scheduler.step(avg_val_loss)
        
        # --- PRINT PERFORMA ---
        print(f"Epoch {epoch+1:3d} | KL: {kl_weight:.5f} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}", end="")

        # --- LOGIKA EARLY STOPPING ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            counter = 0 
            torch.save(model.state_dict(), 'best_bayesian_model.pth')
            print(" ✅ New Best!") 
        else:
            counter += 1
            print(f" ⏳ ({counter}/{patience})") 
            
            if counter >= patience:
                print(f"\n🛑 Early Stopping! Stop di Epoch {epoch+1}.")
                break
    
    # 4. LOAD MODEL TERBAIK 
    print("\n📂 Memuat kembali model dengan Val Loss terbaik...")
    model.load_state_dict(torch.load('best_bayesian_model.pth')) 
    
    return history, model

# ==============================================================================
# 5. MONTE CARLO SIMULATION (Vectorized)
# ==============================================================================
def generate_monte_carlo_predictions(model, dataset, n_samples, device, batch_size=64):
    """
    Vectorized Monte Carlo Sampling dengan TOTAL UNCERTAINTY (Aleatoric + Epistemic).
    """
    model.eval()
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    mc_mus = []
    mc_sigmas = [] 
    
    print(f"\n🎲 Menjalankan Monte Carlo Sampling ({n_samples} Sampel)...")
    
    for _ in tqdm(range(n_samples), desc="Sampling"):
        epoch_mu = []
        epoch_sigma = []
        
        for X_batch, _ in dataloader:
            X_batch = X_batch.to(device)
            with torch.no_grad():
                mu, sigma = model(X_batch) 
                epoch_mu.append(mu.cpu().numpy())
                epoch_sigma.append(sigma.cpu().numpy()) 
                
        mc_mus.append(np.concatenate(epoch_mu))
        mc_sigmas.append(np.concatenate(epoch_sigma))

    # Convert ke Numpy Array (n_samples, n_data, 1)
    mc_mus_array = np.stack(mc_mus)      
    mc_sigmas_array = np.stack(mc_sigmas)
    
    # --- HITUNG TOTAL UNCERTAINTY ---
    
    # 1. Prediksi Mean Akhir (Rata-rata dari semua universe)
    pred_mu = np.mean(mc_mus_array, axis=0)
    
    # 2. Epistemic Uncertainty (Varians dari Mean Prediksi)
    epistemic_variance = np.var(mc_mus_array, axis=0)
    
    # 3. Aleatoric Uncertainty (Rata-rata dari Varians Data)
    aleatoric_variance = np.mean(mc_sigmas_array**2, axis=0)
    
    # 4. Total Uncertainty (Gabungan)
    total_variance = epistemic_variance + aleatoric_variance
    pred_sigma = np.sqrt(total_variance) # Akar dari total varians
    
    # Ambil Data Asli (Actuals)
    actuals = []
    for _, y in dataloader: actuals.append(y.numpy())
    
    return pred_mu, pred_sigma, np.concatenate(actuals)

# ==============================================================================
# 6. EVALUATION FUNCTIONS (Metrics & Plotting)
# ==============================================================================
def calculate_uncertainty_metrics(y_true, y_pred_mean, y_pred_std):
    # PICP (Prediction Interval Coverage Probability)
    lower_bound = y_pred_mean - 1.96 * y_pred_std
    upper_bound = y_pred_mean + 1.96 * y_pred_std
    inside_interval = (y_true >= lower_bound) & (y_true <= upper_bound)
    picp = np.mean(inside_interval)
    
    # MPIW (Mean Prediction Interval Width)
    mpiw = np.mean(upper_bound - lower_bound)
    
    return picp, mpiw, lower_bound, upper_bound

def plot_uncertainty(y_true, y_pred_mean, lower, upper, title="Bayesian Uncertainty Estimation"):
    subset = 150 
    plt.figure(figsize=(12, 6))
    x = np.arange(subset)
    
    if len(y_true) > subset:
        y_true_sub = y_true[-subset:]
        y_pred_sub = y_pred_mean[-subset:]
        lower_sub = lower[-subset:]
        upper_sub = upper[-subset:]
    else:
        y_true_sub = y_true; y_pred_sub = y_pred_mean; lower_sub = lower; upper_sub = upper
        x = np.arange(len(y_true))

    plt.plot(x, y_true_sub, label='Actual Log Return', color='black', alpha=0.7)
    plt.plot(x, y_pred_sub, label='Predicted Mean', color='blue', linestyle='--')
    plt.fill_between(x, lower_sub, upper_sub, color='blue', alpha=0.2, label='95% Confidence Interval')
    plt.title(title); plt.xlabel("Time Steps (Last 150 Hours)"); plt.ylabel("Standardized Log Return")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()

def analyze_portfolio(pred_mu, pred_sigma, actuals, scaler_y, ticker, test_dates, history, risk_free_rate):
    print(f"\n📊 HASIL ANALISIS: {ticker}")
    
    # Inverse Transform
    pred_real = scaler_y.inverse_transform(pred_mu.reshape(-1, 1)).flatten()
    actual_real = scaler_y.inverse_transform(actuals.reshape(-1, 1)).flatten()
    sigma_real = pred_sigma * scaler_y.scale_[0]
    
    # --- METRIK ERROR ---
    rmse = np.sqrt(np.mean((pred_real - actual_real) ** 2))
    mae = np.mean(np.abs(pred_real - actual_real))
    
    numerator = np.abs(pred_real - actual_real)
    denominator = (np.abs(pred_real) + np.abs(actual_real)) / 2 + 1e-8
    smape = np.mean(numerator / denominator) * 100
    print(f"📉 RMSE: {rmse:.6f} | MAE: {mae:.6f} | SMAPE: {smape:.2f}%")
    
    # --- METRIK UNCERTAINTY (PICP & MPIW) ---
    picp, mpiw, lower, upper = calculate_uncertainty_metrics(actuals, pred_mu, pred_sigma)
    print(f"🎲 Uncertainty Metrics: PICP={picp:.4f} (Target 0.95) | MPIW={mpiw:.4f}")
    
    # --- TRADING SIMULATION ---
    portfolio_returns = []
    dynamic_threshold = np.mean(sigma_real)
    print(f"🛡️ Dynamic Risk Threshold (Mean Sigma): {dynamic_threshold:.5f}")
    
    signals = []
    for r_pred, sig, r_act in zip(pred_real, sigma_real, actual_real):
        simple_ret_pred = np.exp(r_pred) - 1
        # Strategy: Long jika Return > 0 AND Uncertainty < Average
        if simple_ret_pred > 0 and sig < dynamic_threshold:
            action = 1.0 # BUY/HOLD
        else:
            action = 0.0 # CASH/SELL
            
        actual_simple_ret = np.exp(r_act) - 1
        portfolio_returns.append(action * actual_simple_ret)
        signals.append(action)
        
    portfolio_returns = np.array(portfolio_returns)

    # --- SHARPE RATIO (Risk Adjusted) ---
    # Konversi Rf tahunan ke Rf per jam (karena data kita hourly)
    rf_hourly = risk_free_rate / (365 * 24)
    
    # Excess Return = Return Portfolio - Risk Free Rate
    excess_returns = portfolio_returns - rf_hourly
    
    avg_excess_ret = np.mean(excess_returns)
    std_excess_ret = np.std(excess_returns)
    
    # Sharpe = Mean(Excess) / Std(Excess) * Sqrt(N)
    sharpe = (avg_excess_ret / (std_excess_ret + 1e-9)) * np.sqrt(8760)
    
    # --- SORTINO RATIO (Downside Risk Only) ---
    # Hanya hitung deviasi standar dari return negatif
    downside_returns = excess_returns[excess_returns < 0]
    
    if len(downside_returns) > 0:
        downside_deviation = np.sqrt(np.mean(downside_returns**2))
        sortino = (avg_excess_ret / (downside_deviation + 1e-9)) * np.sqrt(8760)
    else:
        sortino = np.nan

    print(f"💰 Sharpe Ratio Strategy: {sharpe:.4f}")
    print(f"📉 Sortino Ratio Strategy: {sortino:.4f}")
    print(f"📈 Total Trades Executed: {sum(signals)} hours ({sum(signals)/len(signals)*100:.1f}%)")
    
    
    # --- PLOTTING ---
    plt.figure(figsize=(12, 6))
    if test_dates is not None and len(test_dates) == len(portfolio_returns):
        x_axis = test_dates
    else:
        x_axis = range(len(portfolio_returns))
    
    cum_strategy = np.cumprod(1 + portfolio_returns) - 1
    cum_benchmark = np.cumprod(1 + (np.exp(actual_real)-1)) - 1
    
    plt.plot(x_axis, cum_strategy, label=f'Bayesian Strategy (Sharpe: {sharpe:.2f})', color='#00C853', linewidth=2)
    plt.plot(x_axis, cum_benchmark, label='Buy & Hold (Benchmark)', color='gray', linestyle='--', alpha=0.6)
    plt.title(f"Cumulative Returns Comparison: {ticker}"); plt.xlabel("Date"); plt.ylabel("Cumulative Return")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()
    
    if history:
        plt.figure(figsize=(10, 3))
        plt.plot(history['train_loss'], label='Train Loss (ELBO)'); plt.plot(history['val_loss'], label='Val Loss (NLL)')
        plt.title("Training Loss History"); plt.legend(); plt.grid(True, alpha=0.3); plt.show()
        
    # Plot Uncertainty
    plot_uncertainty(actuals, pred_mu, lower, upper, title=f"Bayesian Uncertainty ({ticker})")


In [ ]:
# ==============================================================================
# 7. MAIN EXECUTION FLOW
# ==============================================================================
if __name__ == "__main__":
    print(f"⚙️ Running on: {device}")
    
    if os.path.exists(MODEL_PATH) and os.path.exists(ARTIFACTS_PATH):
        print("\n📂 MENGGUNAKAN MODEL TERSIMPAN...")
        print("   (Hapus file .pth dan .pkl jika ingin training ulang)")
        
        with open(ARTIFACTS_PATH, 'rb') as f: artifacts = pickle.load(f)
        
        raw_preds = artifacts['monte_carlo_results']
        scaler_y = artifacts['scaler_y']
        saved_config = artifacts['config']
        test_dates = artifacts.get('test_dates')
        history = artifacts.get('history')
        
        analyze_portfolio(
            *raw_preds, scaler_y, saved_config['ticker'], test_dates, history, saved_config.get('risk_free_rate', 0.0)
        )
        
    else:
        print("\n🚀 MEMULAI TRAINING BARU...")
        try:
            # 1. Download & Preprocess
            df = download_and_preprocess_data(CONFIG['ticker'], CONFIG['start_date'], CONFIG['end_date'])
            print("Feature Engineering...")
            df_processed = feature_engineering(df)
            
            # 2. Prepare Data
            feature_cols = [c for c in df_processed.columns if c != 'log_return']
            X_raw = df_processed[feature_cols].values
            y_raw = df_processed['log_return'].values.reshape(-1, 1)
            
            # Cari index pemisah berdasarkan tanggal CONFIG['test_start_date']
            split_date_ts = pd.Timestamp(CONFIG['test_start_date'])
            # Pastikan index sudah datetime
            if not isinstance(df_processed.index, pd.DatetimeIndex):
                df_processed.index = pd.to_datetime(df_processed.index)
                
            # Filter masker
            train_mask = df_processed.index < split_date_ts
            test_mask = df_processed.index >= split_date_ts
            
            split_idx = np.sum(train_mask)
            
            print(f"✂️ Splitting Data pada: {CONFIG['test_start_date']}")
            print(f"   Train Size: {split_idx} jam | Test Size: {len(df_processed) - split_idx} jam")
            
            test_dates = df_processed.index[split_idx + CONFIG['seq_length']:] 
            
            scaler_X, scaler_y = StandardScaler(), StandardScaler()
            X_train = scaler_X.fit_transform(X_raw[:split_idx])
            X_test = scaler_X.transform(X_raw[split_idx:])
            y_train = scaler_y.fit_transform(y_raw[:split_idx])
            y_test = scaler_y.transform(y_raw[split_idx:])
            
            train_ds = CryptoDataset(X_train, y_train, CONFIG['seq_length'])
            test_ds = CryptoDataset(X_test, y_test, CONFIG['seq_length'])
            train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=False)
            test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False)
            
            # 3. Model & Training
            model = BayesianLSTMModel(len(feature_cols), CONFIG['hidden_size_1'], CONFIG['hidden_size_2']).to(device)
            history, model = train_engine(model, train_loader, test_loader, CONFIG['epochs'], device, CONFIG['lr'])
            
            # 4. Monte Carlo Sampling
            mc_mu, mc_sigma, mc_actuals = generate_monte_carlo_predictions(
                model, test_ds, CONFIG['n_samples'], device, batch_size=CONFIG['batch_size']
            )
            raw_predictions = (mc_mu, mc_sigma, mc_actuals)
            
            # 5. Save Artifacts
            print("\n💾 Menyimpan Model & Hasil Simulasi...")
            torch.save(model.state_dict(), MODEL_PATH)
            with open(ARTIFACTS_PATH, 'wb') as f:
                pickle.dump({
                    'config': CONFIG, 'scaler_X': scaler_X, 'scaler_y': scaler_y,
                    'input_dim': len(feature_cols), 'monte_carlo_results': raw_predictions, 
                    'history': history, 'test_dates': test_dates
                }, f)
            
            # 6. Analisis
            analyze_portfolio(*raw_predictions, scaler_y, CONFIG['ticker'], test_dates, history, CONFIG['risk_free_rate'])
            
        except Exception as e:
            print(f"\n❌ Terjadi Kesalahan: {e}")
            import traceback
            traceback.print_exc()